# Agentick PPO Benchmark — Success Rate Dashboard

Visualizes PPO training results, separated by **render mode** (isometric vs flat) and **reward mode** (dense vs sparse).  
All metrics focus on **success rate** (0–1).

In [ ]:
import json
import warnings
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import numpy as np

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
})

PALETTE = [
    "#E69F00", "#56B4E9", "#009E73", "#F0E442",
    "#0072B2", "#D55E00", "#CC79A7", "#000000",
]
DIFFICULTIES = ["easy", "medium", "hard", "expert"]
DIFF_COLORS = {"easy": "#009E73", "medium": "#0072B2", "hard": "#E69F00", "expert": "#D55E00"}
ALL_CATEGORIES = [
    "navigation", "planning", "reasoning", "memory",
    "generalization", "multi_agent", "resource", "timing",
    "deception", "curiosity",
]

print("Imports ready.")

## 1. Load all results (separated by render mode)

In [ ]:
def find_project_root():
    p = Path.cwd()
    for _ in range(10):
        if (p / "results").is_dir():
            return p
        p = p.parent
    raise FileNotFoundError("Cannot find results/ directory.")

ROOT = find_project_root()
RESULTS_DIR = ROOT / "results"
print(f"Project root: {ROOT}")
print(f"Results dir:  {RESULTS_DIR}")


def _detect_render_mode(run_path):
    """Detect render mode from config.yaml or training_summary.json."""
    # New runs store it in training_summary.json
    ts_path = run_path / "training_summary.json"
    if ts_path.exists():
        with open(ts_path) as f:
            ts = json.load(f)
        if "render_mode" in ts:
            return ts["render_mode"]
    # Fall back to config.yaml
    cfg_path = run_path / "config.yaml"
    if cfg_path.exists():
        import re
        text = cfg_path.read_text()
        m = re.search(r"render_modes:\s*\[([^\]]+)\]", text)
        if m:
            modes = [s.strip().strip("'\"") for s in m.group(1).split(",")]
            return modes[0] if modes else "rgb_array"
    # Fall back: check run name
    name = run_path.name.lower()
    if "flat" in name:
        return "rgb_array_flat"
    return "rgb_array"


def _compute_sr_curve_from_npz(eval_dir):
    """Compute success rate over training from evaluations.npz.
    
    Uses heuristic: episode return > 0 means success.
    """
    npz_path = eval_dir / "evaluations.npz"
    if not npz_path.exists():
        return []
    data = np.load(str(npz_path))
    timesteps = data["timesteps"]
    results = data["results"]  # (n_evals, n_episodes)
    curve = []
    for i, ts in enumerate(timesteps):
        ep_returns = results[i]
        curve.append({
            "timestep": int(ts),
            "success_rate": float(np.mean(ep_returns > 0)),
        })
    return curve


# --- Discover all benchmark runs ---
benchmarks = {}  # run_name -> {...}

for run_dir in sorted(RESULTS_DIR.rglob("training_summary.json")):
    run_path = run_dir.parent
    run_name = run_path.name
    data = {"path": run_path}

    # Load training summary
    with open(run_dir) as f:
        data["training_summary"] = json.load(f)

    # Detect render mode
    data["render_mode"] = _detect_render_mode(run_path)

    # Load metadata
    meta_path = run_path / "metadata.json"
    if meta_path.exists():
        with open(meta_path) as f:
            data["metadata"] = json.load(f)

    # Load summary
    summary_path = run_path / "summary.json"
    if summary_path.exists():
        with open(summary_path) as f:
            data["summary"] = json.load(f)

    # Load per-task metrics + compute SR training curves from evaluations.npz
    per_task = {}
    per_task_dir = run_path / "per_task"
    if per_task_dir.exists():
        for metrics_file in per_task_dir.rglob("metrics.json"):
            with open(metrics_file) as f:
                m = json.load(f)
            key = f"{m['task']}_{m['difficulty']}"

            # Compute SR training curve from evaluations.npz
            eval_dir = metrics_file.parent / "eval"
            sr_curve = []
            # First try: use success_rate from training_curve if available (new runs)
            tc = m.get("training_curve", [])
            if tc and "success_rate" in tc[0]:
                sr_curve = [{"timestep": p["timestep"], "success_rate": p["success_rate"]} for p in tc]
            else:
                # Fall back: compute from evaluations.npz
                sr_curve = _compute_sr_curve_from_npz(eval_dir)
            m["sr_curve"] = sr_curve
            per_task[key] = m
    data["per_task"] = per_task
    benchmarks[run_name] = data

print(f"\nFound {len(benchmarks)} benchmark run(s):")
for name, d in benchmarks.items():
    n_metrics = len(d["per_task"])
    reward_mode = d["training_summary"].get("reward_mode", "unknown")
    render_mode = d["render_mode"]
    print(f"  {name}")
    print(f"    reward={reward_mode}, render={render_mode}, {n_metrics} task-difficulty combos")

In [ ]:
# --- Build unified records with render_mode ---
all_records = []
for run_name, bdata in benchmarks.items():
    reward_mode = bdata["training_summary"].get("reward_mode", "unknown")
    render_mode = bdata["render_mode"]
    for key, m in bdata["per_task"].items():
        rec = {
            "run": run_name,
            "reward_mode": reward_mode,
            "render_mode": render_mode,
            "render_label": "flat" if "flat" in render_mode else "isometric",
            "task": m["task"],
            "task_short": m["task"].replace("-v0", ""),
            "difficulty": m["difficulty"],
            "category": m.get("category", "unknown"),
            "success_rate": m["success_rate"],
            "mean_return": m["mean_return"],
            "std_return": m["std_return"],
            "mean_length": m.get("mean_length", np.nan),
            "training_curve": m.get("training_curve", []),
            "sr_curve": m.get("sr_curve", []),
        }
        all_records.append(rec)

# Utility lookups
tasks_all = sorted(set(r["task"] for r in all_records))
categories_all = sorted(set(r["category"] for r in all_records))
runs_all = sorted(benchmarks.keys())
render_modes_all = sorted(set(r["render_label"] for r in all_records))

# Short labels: "PPO (dense, flat)" etc.
run_labels = {}
for rn in runs_all:
    reward = benchmarks[rn]["training_summary"].get("reward_mode", rn)
    render = "flat" if "flat" in benchmarks[rn]["render_mode"] else "iso"
    run_labels[rn] = f"PPO ({reward}, {render})"

# Colors per run
run_colors = {rn: PALETTE[i % len(PALETTE)] for i, rn in enumerate(runs_all)}
# Colors per category
cat_colors = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(categories_all)}

print(f"{len(all_records)} records across {len(tasks_all)} tasks, {len(runs_all)} runs.")
print(f"Render modes: {render_modes_all}")
print(f"Runs: {list(run_labels.values())}")

---
## 2. Summary Table

In [ ]:
header = f"{'Run':<35} {'Render':<12} {'Tasks':>6} {'Mean SR':>8} {'Best SR':>8}"
print(header)
print("-" * len(header))
for rn in runs_all:
    recs = [r for r in all_records if r["run"] == rn]
    mean_sr = np.mean([r["success_rate"] for r in recs])
    best_sr = max(r["success_rate"] for r in recs)
    n_tasks = len(set(r["task"] for r in recs))
    render = recs[0]["render_label"]
    print(f"{run_labels[rn]:<35} {render:<12} {n_tasks:>6} {mean_sr:>8.1%} {best_sr:>8.1%}")

---
## 3. Success Rate Heatmap (Tasks x Difficulties) — per run

In [ ]:
for rn in runs_all:
    recs = [r for r in all_records if r["run"] == rn]

    # Group by category for ordering
    cat_tasks = defaultdict(list)
    for r in recs:
        if r["task"] not in cat_tasks[r["category"]]:
            cat_tasks[r["category"]].append(r["task"])
    for c in cat_tasks:
        cat_tasks[c].sort()

    ordered_tasks = []
    cat_breaks = []
    cat_labels_list = []
    for cat in ALL_CATEGORIES:
        if cat in cat_tasks and cat_tasks[cat]:
            cat_breaks.append(len(ordered_tasks))
            cat_labels_list.append(cat)
            ordered_tasks.extend(cat_tasks[cat])
    for cat in sorted(cat_tasks.keys()):
        if cat not in ALL_CATEGORIES and cat_tasks[cat]:
            cat_breaks.append(len(ordered_tasks))
            cat_labels_list.append(cat)
            ordered_tasks.extend(cat_tasks[cat])

    lookup = {(r["task"], r["difficulty"]): r for r in recs}
    n_t, n_d = len(ordered_tasks), len(DIFFICULTIES)
    matrix = np.full((n_t, n_d), np.nan)
    for i, t in enumerate(ordered_tasks):
        for j, d in enumerate(DIFFICULTIES):
            if (t, d) in lookup:
                matrix[i, j] = lookup[(t, d)]["success_rate"]

    fig_h = max(8, n_t * 0.32)
    fig, ax = plt.subplots(figsize=(6, fig_h))
    im = ax.imshow(matrix, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1, interpolation="nearest")

    ax.set_xticks(range(n_d))
    ax.set_xticklabels([d.capitalize() for d in DIFFICULTIES])
    ax.set_yticks(range(n_t))
    ax.set_yticklabels([t.replace("-v0", "") for t in ordered_tasks], fontsize=7)

    for i in range(n_t):
        for j in range(n_d):
            val = matrix[i, j]
            if not np.isnan(val):
                color = "white" if val < 0.35 or val > 0.85 else "black"
                ax.text(j, i, f"{val:.0%}", ha="center", va="center", fontsize=6, color=color, fontweight="bold")

    for brk in cat_breaks[1:]:
        ax.axhline(y=brk - 0.5, color="black", linewidth=1.2)

    for idx, (brk, label) in enumerate(zip(cat_breaks, cat_labels_list)):
        nxt = cat_breaks[idx + 1] if idx + 1 < len(cat_breaks) else n_t
        mid = (brk + nxt - 1) / 2
        ax.text(n_d + 0.2, mid, label.replace("_", " ").title(),
                va="center", fontsize=7, fontstyle="italic", color="#555")

    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.12)
    cbar.set_label("Success Rate")
    ax.set_title(f"Success Rate — {run_labels[rn]}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Difficulty")
    plt.tight_layout()
    plt.show()

---
## 4. Success Rate Training Curves — per task gallery

Shows how success rate evolves during training (computed from eval returns > 0).  
Each subplot is one task, with 4 difficulty curves overlaid. Y-axis is 0–1.

In [ ]:
for rn in runs_all:
    recs = [r for r in all_records if r["run"] == rn and r["sr_curve"]]
    tasks_with_curves = sorted(set(r["task"] for r in recs))

    n = len(tasks_with_curves)
    if n == 0:
        print(f"No SR curves for {run_labels[rn]}")
        continue

    n_cols = 4
    n_rows = (n + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows), squeeze=False)

    for ti, task in enumerate(tasks_with_curves):
        ax = axes[ti // n_cols][ti % n_cols]
        task_recs = [r for r in recs if r["task"] == task]

        for r in task_recs:
            curve = r["sr_curve"]
            ts = [p["timestep"] for p in curve]
            srs = [p["success_rate"] for p in curve]
            color = DIFF_COLORS.get(r["difficulty"], "#999")
            ax.plot(ts, srs, label=r["difficulty"], color=color, linewidth=1.2)

        ax.set_title(task.replace("-v0", ""), fontsize=8, fontweight="bold")
        ax.set_ylim(-0.05, 1.05)
        ax.tick_params(labelsize=6)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
        if ti == 0:
            ax.legend(fontsize=5)

    for idx in range(n, n_rows * n_cols):
        axes[idx // n_cols][idx % n_cols].set_visible(False)

    fig.suptitle(f"Training SR Curves (all difficulties) — {run_labels[rn]}",
                 fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

---
## 5. Eval SR Curves by Category

Medium difficulty, grouped by category. Y-axis is 0–1.

In [ ]:
for rn in runs_all:
    recs = [r for r in all_records if r["run"] == rn and r["difficulty"] == "medium" and r["sr_curve"]]
    if not recs:
        recs = [r for r in all_records if r["run"] == rn and r["difficulty"] == "easy" and r["sr_curve"]]

    cat_recs = defaultdict(list)
    for r in recs:
        cat_recs[r["category"]].append(r)

    active_cats = sorted(cat_recs.keys())
    if not active_cats:
        print(f"No SR curves for {run_labels[rn]}")
        continue

    n_cols = min(3, len(active_cats))
    n_rows = (len(active_cats) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows), squeeze=False)

    for idx, cat in enumerate(active_cats):
        ax = axes[idx // n_cols][idx % n_cols]
        for ti, r in enumerate(sorted(cat_recs[cat], key=lambda x: x["task"])):
            curve = r["sr_curve"]
            ts = [p["timestep"] for p in curve]
            srs = [p["success_rate"] for p in curve]
            color = PALETTE[ti % len(PALETTE)]
            ax.plot(ts, srs, label=r["task_short"], color=color, linewidth=1.2)

        ax.set_title(cat.replace("_", " ").title(), fontsize=10, fontweight="bold")
        ax.set_xlabel("Timesteps")
        ax.set_ylabel("Success Rate")
        ax.set_ylim(-0.05, 1.05)
        ax.legend(fontsize=5, loc="best", ncol=2)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))

    for idx in range(len(active_cats), n_rows * n_cols):
        axes[idx // n_cols][idx % n_cols].set_visible(False)

    fig.suptitle(f"Eval SR Curves by Category (medium) — {run_labels[rn]}",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

---
## 6. Per-Task Success Rate Bar Chart

In [ ]:
for rn in runs_all:
    recs = [r for r in all_records if r["run"] == rn]

    task_stats = defaultdict(lambda: {"sr": [], "cat": "unknown"})
    for r in recs:
        task_stats[r["task"]]["sr"].append(r["success_rate"])
        task_stats[r["task"]]["cat"] = r["category"]

    sorted_tasks = sorted(task_stats.keys(), key=lambda t: np.mean(task_stats[t]["sr"]))

    fig, ax = plt.subplots(figsize=(8, max(5, len(sorted_tasks) * 0.3)))

    names = [t.replace("-v0", "") for t in sorted_tasks]
    sr_means = [np.mean(task_stats[t]["sr"]) for t in sorted_tasks]
    sr_stds = [np.std(task_stats[t]["sr"]) for t in sorted_tasks]
    colors = [cat_colors.get(task_stats[t]["cat"], "#999") for t in sorted_tasks]

    ax.barh(range(len(names)), sr_means, xerr=sr_stds, color=colors,
            edgecolor="white", linewidth=0.5, capsize=2)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=7)
    ax.set_xlabel("Mean Success Rate")
    ax.set_xlim(0, 1.05)
    ax.axvline(x=np.mean(sr_means), color="red", linestyle="--", alpha=0.5,
               label=f"Overall: {np.mean(sr_means):.1%}")
    ax.legend(fontsize=7)
    ax.set_title(f"Per-Task Success Rate (avg over difficulties) — {run_labels[rn]}",
                 fontsize=10, fontweight="bold")

    # Category legend
    used_cats = sorted(set(task_stats[t]["cat"] for t in sorted_tasks))
    patches = [Patch(facecolor=cat_colors.get(c, "#999"), label=c.replace("_", " ").title())
               for c in used_cats]
    ax.legend(handles=patches, fontsize=6, loc="lower right")

    plt.tight_layout()
    plt.show()

---
## 7. Difficulty Scaling (Success Rate)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

for rn in runs_all:
    recs = [r for r in all_records if r["run"] == rn]
    color = run_colors[rn]
    label = run_labels[rn]

    sr_by_diff, sr_err = [], []
    for d in DIFFICULTIES:
        d_recs = [r for r in recs if r["difficulty"] == d]
        srs = [r["success_rate"] for r in d_recs]
        sr_by_diff.append(np.mean(srs) if srs else 0)
        sr_err.append(np.std(srs) if srs else 0)

    ax.errorbar(DIFFICULTIES, sr_by_diff, yerr=sr_err, marker="o", linewidth=2,
                markersize=8, capsize=5, label=label, color=color)

ax.set_ylabel("Mean Success Rate")
ax.set_xlabel("Difficulty")
ax.set_ylim(-0.05, 1.05)
ax.set_title("Success Rate vs Difficulty", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

---
## 8. Category Spider / Radar Charts (SR)

In [ ]:
n_runs = len(runs_all)
fig, axes = plt.subplots(1, n_runs, figsize=(7 * n_runs, 6),
                          subplot_kw=dict(projection="polar"))
if n_runs == 1:
    axes = [axes]

for ax, rn in zip(axes, runs_all):
    recs = [r for r in all_records if r["run"] == rn]
    color = run_colors[rn]

    cat_scores = defaultdict(list)
    for r in recs:
        cat_scores[r["category"]].append(r["success_rate"])

    active_cats = [c for c in ALL_CATEGORIES if c in cat_scores]
    for c in sorted(cat_scores.keys()):
        if c not in active_cats:
            active_cats.append(c)

    if len(active_cats) < 3:
        ax.set_title(f"{run_labels[rn]}\n(too few categories)")
        continue

    labels = [c.replace("_", " ").title() for c in active_cats]
    values = [np.mean(cat_scores[c]) for c in active_cats]

    angles = np.linspace(0, 2 * np.pi, len(active_cats), endpoint=False).tolist()
    values_plot = values + values[:1]
    angles_plot = angles + angles[:1]

    ax.plot(angles_plot, values_plot, "o-", linewidth=2, color=color)
    ax.fill(angles_plot, values_plot, alpha=0.2, color=color)
    ax.set_xticks(angles)
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title(f"Capability Radar\n{run_labels[rn]}", pad=20, fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

---
## 9. Cross-Run Comparison — Side-by-Side SR Bars

Compares all runs on the same tasks (averaged over difficulties).

In [ ]:
if len(runs_all) >= 2:
    # Build per-task per-run SR
    task_run_sr = defaultdict(dict)  # task -> {run: mean_sr}
    for r in all_records:
        task_run_sr[r["task"]].setdefault(r["run"], []).append(r["success_rate"])

    # Find tasks present in all runs
    common_tasks = sorted([
        t for t in task_run_sr
        if all(rn in task_run_sr[t] for rn in runs_all)
    ])

    if common_tasks:
        # Sort by mean SR of first run
        common_tasks.sort(key=lambda t: np.mean(task_run_sr[t][runs_all[0]]))

        fig, ax = plt.subplots(figsize=(10, max(5, len(common_tasks) * 0.35)))
        y = np.arange(len(common_tasks))
        n_runs_cmp = len(runs_all)
        height = 0.8 / n_runs_cmp

        for ri, rn in enumerate(runs_all):
            vals = [np.mean(task_run_sr[t][rn]) for t in common_tasks]
            offset = (ri - n_runs_cmp / 2 + 0.5) * height
            ax.barh(y + offset, vals, height, label=run_labels[rn],
                    color=run_colors[rn], edgecolor="white", linewidth=0.5)

        ax.set_yticks(y)
        ax.set_yticklabels([t.replace("-v0", "") for t in common_tasks], fontsize=7)
        ax.set_xlabel("Mean Success Rate")
        ax.set_xlim(0, 1.05)
        ax.set_title("Per-Task Success Rate Comparison", fontsize=12, fontweight="bold")
        ax.legend(loc="lower right", fontsize=7)
        plt.tight_layout()
        plt.show()
else:
    print("Only one run — skipping comparison.")

---
## 10. Category-Level SR Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

n_runs_plot = len(runs_all)
bar_width = 0.8 / max(n_runs_plot, 1)

for ri, rn in enumerate(runs_all):
    recs = [r for r in all_records if r["run"] == rn]
    cat_sr = defaultdict(list)
    for r in recs:
        cat_sr[r["category"]].append(r["success_rate"])

    cats = sorted(cat_sr.keys())
    x = np.arange(len(cats))
    offset = (ri - n_runs_plot / 2 + 0.5) * bar_width

    sr_vals = [np.mean(cat_sr[c]) for c in cats]
    sr_errs = [np.std(cat_sr[c]) for c in cats]
    ax.bar(x + offset, sr_vals, bar_width, yerr=sr_errs, label=run_labels[rn],
           color=run_colors[rn], capsize=3, edgecolor="white", linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels([c.replace("_", " ").title() for c in cats], rotation=35, ha="right")
ax.set_ylabel("Mean Success Rate")
ax.set_ylim(0, 1.05)
ax.set_title("Success Rate by Category", fontweight="bold")
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

---
## 11. Category x Difficulty Heatmap (SR)

In [ ]:
for rn in runs_all:
    recs = [r for r in all_records if r["run"] == rn]
    cats = sorted(set(r["category"] for r in recs))
    matrix = np.full((len(cats), len(DIFFICULTIES)), np.nan)

    for ci, cat in enumerate(cats):
        for di, diff in enumerate(DIFFICULTIES):
            vals = [r["success_rate"] for r in recs if r["category"] == cat and r["difficulty"] == diff]
            if vals:
                matrix[ci, di] = np.mean(vals)

    fig, ax = plt.subplots(figsize=(6, max(3, len(cats) * 0.5)))
    im = ax.imshow(matrix, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1, interpolation="nearest")

    ax.set_xticks(range(len(DIFFICULTIES)))
    ax.set_xticklabels([d.capitalize() for d in DIFFICULTIES])
    ax.set_yticks(range(len(cats)))
    ax.set_yticklabels([c.replace("_", " ").title() for c in cats], fontsize=9)

    for i in range(len(cats)):
        for j in range(len(DIFFICULTIES)):
            val = matrix[i, j]
            if not np.isnan(val):
                color = "white" if val < 0.35 or val > 0.85 else "black"
                ax.text(j, i, f"{val:.0%}", ha="center", va="center", fontsize=10,
                        color=color, fontweight="bold")

    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.04)
    cbar.set_label("Mean Success Rate")
    ax.set_title(f"Category x Difficulty — {run_labels[rn]}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Difficulty")
    plt.tight_layout()
    plt.show()

---
## 12. Top / Bottom Performers

In [ ]:
K = 10

for rn in runs_all:
    recs = [r for r in all_records if r["run"] == rn]
    task_avg = defaultdict(lambda: {"sr": [], "cat": "unknown"})
    for r in recs:
        task_avg[r["task"]]["sr"].append(r["success_rate"])
        task_avg[r["task"]]["cat"] = r["category"]

    task_sr = {t: np.mean(v["sr"]) for t, v in task_avg.items()}
    sorted_by_sr = sorted(task_sr.items(), key=lambda x: x[1], reverse=True)

    top_k = sorted_by_sr[:K]
    bottom_k = sorted_by_sr[-K:]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    names_top = [t.replace("-v0", "") for t, _ in top_k]
    vals_top = [v for _, v in top_k]
    cols_top = [cat_colors.get(task_avg[t]["cat"], "#999") for t, _ in top_k]
    ax1.barh(range(len(names_top)), vals_top, color=cols_top, edgecolor="white", linewidth=0.5)
    ax1.set_yticks(range(len(names_top)))
    ax1.set_yticklabels(names_top, fontsize=8)
    ax1.set_xlim(0, 1.05)
    ax1.set_xlabel("Mean Success Rate")
    ax1.set_title(f"Top {K} Tasks", fontsize=10, fontweight="bold", color="#009E73")
    ax1.invert_yaxis()

    names_bot = [t.replace("-v0", "") for t, _ in bottom_k]
    vals_bot = [v for _, v in bottom_k]
    cols_bot = [cat_colors.get(task_avg[t]["cat"], "#999") for t, _ in bottom_k]
    ax2.barh(range(len(names_bot)), vals_bot, color=cols_bot, edgecolor="white", linewidth=0.5)
    ax2.set_yticks(range(len(names_bot)))
    ax2.set_yticklabels(names_bot, fontsize=8)
    ax2.set_xlim(0, 1.05)
    ax2.set_xlabel("Mean Success Rate")
    ax2.set_title(f"Bottom {K} Tasks", fontsize=10, fontweight="bold", color="#D55E00")
    ax2.invert_yaxis()

    patches = [Patch(facecolor=cat_colors.get(c, "#999"), label=c.replace("_", " ").title())
               for c in sorted(set(task_avg[t]["cat"] for t in task_avg))]
    fig.legend(handles=patches, fontsize=6, loc="lower center", ncol=len(patches),
              bbox_to_anchor=(0.5, -0.05))

    fig.suptitle(f"Top & Bottom Performers — {run_labels[rn]}", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

---
## 13. Cross-Run SR Training Curve Comparison

Overlays SR curves from different runs for the same task (medium difficulty).

In [ ]:
if len(runs_all) >= 2:
    diff_target = "medium"

    curves_by_run = {}
    for rn in runs_all:
        curves_by_run[rn] = {}
        for r in all_records:
            if r["run"] == rn and r["difficulty"] == diff_target and r["sr_curve"]:
                curves_by_run[rn][r["task"]] = r

    common = sorted(set.intersection(*[set(curves_by_run[rn].keys()) for rn in runs_all]))

    if common:
        n = len(common)
        n_cols = 4
        n_rows = (n + n_cols - 1) // n_cols
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows), squeeze=False)

        for ti, task in enumerate(common):
            ax = axes[ti // n_cols][ti % n_cols]
            for rn in runs_all:
                r = curves_by_run[rn][task]
                curve = r["sr_curve"]
                ts = [p["timestep"] for p in curve]
                srs = [p["success_rate"] for p in curve]
                color = run_colors[rn]
                ax.plot(ts, srs, label=run_labels[rn], color=color, linewidth=1.3)

            ax.set_title(task.replace("-v0", ""), fontsize=8, fontweight="bold")
            ax.set_ylim(-0.05, 1.05)
            ax.tick_params(labelsize=6)
            ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
            if ti == 0:
                ax.legend(fontsize=5)

        for idx in range(n, n_rows * n_cols):
            axes[idx // n_cols][idx % n_cols].set_visible(False)

        fig.suptitle(f"Cross-Run SR Curves ({diff_target} difficulty)",
                     fontsize=13, fontweight="bold", y=1.01)
        plt.tight_layout()
        plt.show()
    else:
        print("No common tasks with SR curves across runs.")
else:
    print("Only one run — skipping cross-run comparison.")

---
## 14. Aggregate Stats Summary

In [ ]:
for rn in runs_all:
    recs = [r for r in all_records if r["run"] == rn]
    all_sr = [r["success_rate"] for r in recs]
    n_solved = sum(1 for s in all_sr if s > 0)
    n_perfect = sum(1 for s in all_sr if s >= 1.0)
    n_total = len(all_sr)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    # Distribution of success rates
    ax1.hist(all_sr, bins=20, color=PALETTE[4], edgecolor="white", linewidth=0.5, range=(0, 1))
    ax1.set_xlabel("Success Rate")
    ax1.set_ylabel("Count")
    ax1.set_title("Distribution of Success Rates", fontweight="bold")
    ax1.axvline(x=np.mean(all_sr), color="red", linestyle="--",
                label=f"Mean: {np.mean(all_sr):.2%}")
    ax1.legend(fontsize=7)

    # Pie chart
    pie_data = [n_perfect, n_solved - n_perfect, n_total - n_solved]
    pie_labels = [f"Perfect (SR=100%)\n{n_perfect}",
                  f"Partial (0<SR<100%)\n{n_solved - n_perfect}",
                  f"Unsolved (SR=0%)\n{n_total - n_solved}"]
    pie_colors = ["#009E73", "#E69F00", "#D55E00"]
    nonzero = [(d, l, c) for d, l, c in zip(pie_data, pie_labels, pie_colors) if d > 0]
    if nonzero:
        ax2.pie([d for d, _, _ in nonzero], labels=[l for _, l, _ in nonzero],
                colors=[c for _, _, c in nonzero], autopct="%1.0f%%", startangle=90,
                textprops={"fontsize": 8})
    ax2.set_title(f"Task-Difficulty Combos ({n_total} total)", fontweight="bold")

    fig.suptitle(f"Aggregate — {run_labels[rn]}", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

    print(f"\n{run_labels[rn]}:")
    print(f"  Render mode:  {benchmarks[rn]['render_mode']}")
    print(f"  Total combos: {n_total}")
    print(f"  Perfect:      {n_perfect} ({n_perfect/n_total:.0%})")
    print(f"  Partial:      {n_solved - n_perfect} ({(n_solved-n_perfect)/n_total:.0%})")
    print(f"  Unsolved:     {n_total - n_solved} ({(n_total-n_solved)/n_total:.0%})")
    print(f"  Mean SR:      {np.mean(all_sr):.2%}")

---
## 15. Benchmark Metadata

In [ ]:
for rn in runs_all:
    meta = benchmarks[rn].get("metadata", {})
    ts_data = benchmarks[rn].get("training_summary", {})
    summary = benchmarks[rn].get("summary", {})

    print(f"{'='*60}")
    print(f"Run: {rn}")
    print(f"{'='*60}")
    print(f"  Render mode:   {benchmarks[rn]['render_mode']}")
    print(f"  Reward mode:   {ts_data.get('reward_mode', 'N/A')}")
    if meta:
        print(f"  Timestamp:     {meta.get('timestamp', 'N/A')}")
        print(f"  CUDA Device:   {meta.get('cuda_device', 'N/A')}")
        print(f"  Git hash:      {meta.get('git_hash', 'N/A')[:12]}")
    print(f"  Timesteps:     {ts_data.get('total_timesteps', 'N/A'):,}")
    if summary:
        total_secs = summary.get('total_time_seconds', 0)
        print(f"  Total time:    {total_secs / 3600:.1f} hours")
    print()